# 03 — Submission
Memuat model dari `models/pipeline_model.pkl`, memprediksi `test.csv`, dan menulis
`submissions/submission.csv` (label **angka 1–5**, urutan baris sama dengan `sample_submission.csv`).

Di Kaggle file juga disalin ke `/kaggle/working/submission.csv`. Jika dijalankan di sesi terpisah dari
`02_modeling.ipynb`, tambahkan output notebook 02 (berisi file `.pkl`) sebagai *Input*; `find_model` akan
mencarinya di `/kaggle/input`.

In [ ]:
# === Setup: lokasi repo (Kaggle: selalu clone versi terbaru dari GitHub) ===
import sys, subprocess, shutil
from pathlib import Path

REPO_URL = "https://github.com/yafifaldin/structural-damage-classification.git"
ON_KAGGLE = Path("/kaggle/working").exists()

ROOT = None
if ON_KAGGLE:
    ROOT = Path("/kaggle/working/repo")
    shutil.rmtree(ROOT, ignore_errors=True)
    try:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
        print(subprocess.run(["git", "-C", str(ROOT), "log", "-1", "--oneline"],
                             capture_output=True, text=True).stdout)
    except Exception as exc:  # tanpa Internet: cari repo yang di-upload sebagai Kaggle Dataset
        print("git clone gagal (Internet off?):", exc)
        hits = sorted(Path("/kaggle/input").glob("**/scripts/__init__.py"))
        ROOT = hits[0].parent.parent if hits else None
else:
    for cand in (Path.cwd(), Path.cwd().parent):
        if (cand / "scripts" / "__init__.py").exists():
            ROOT = cand.resolve()
            break
assert ROOT is not None, "Folder repo (berisi scripts/) tidak ditemukan."

for name in [m for m in sys.modules if m == "scripts" or m.startswith("scripts.")]:
    del sys.modules[name]          # pastikan modul versi terbaru yang dipakai
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

In [ ]:
import pandas as pd
from scripts.preprocess import load_data, TARGET, ID_COL
from scripts.predict import load_model, predict_grades, format_submission, save_submission

model = load_model()
_, test, sample = load_data()
sub = format_submission(sample, test[ID_COL], predict_grades(model, test))
path = save_submission(sub)
print("tersimpan:", path, sub.shape)
sub.head()

In [ ]:
assert list(sub.columns) == list(sample.columns)
assert len(sub) == len(sample) and (sub[ID_COL].to_numpy() == sample[ID_COL].to_numpy()).all()
assert sub[TARGET].between(1, 5).all()
print("format OK")
print(sub[TARGET].value_counts(normalize=True).sort_index().round(3))